In [1]:
import torch                                              # core PyTorch library (tensors, autograd, GPU)
import torch.nn as nn                                     # neural network layers (Linear, Conv2d, etc.)
import torch.optim as optim                               # optimisers (Adam, SGD, etc.)
from torch.utils.data import DataLoader                   # batches + shuffling for datasets
from torchvision import datasets, transforms, models       # MNIST dataset, transforms, and famous models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # choose GPU if available else CPU


transform = transforms.Compose([                           # chain multiple preprocessing steps
    transforms.ToTensor(),                                 # convert PIL image -> float tensor [0,1] with shape (1,28,28)
    transforms.Normalize((0.1307,), (0.3081,))              # normalise using MNIST mean/std for faster/stabler learning
])                                                         # end transform pipeline

train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)   # load/download MNIST training set
test_ds  = datasets.MNIST(root=".", train=False, download=True, transform=transform)  # load/download MNIST test set

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)  # training batches
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True) # test batches


def get_model(name: str) -> nn.Module:                    # factory function: returns a model based on a string name
    name = name.lower()                                   # make the selection case-insensitive (e.g., "ResNet18" works)

    if name == "mlp":                                     # option 1: simple fully-connected baseline
        return nn.Sequential(                              # Sequential stacks layers in order
            nn.Flatten(),                                  # (N,1,28,28) -> (N,784)
            nn.Linear(28 * 28, 512),                       # 784 -> 512 dense layer
            nn.ReLU(),                                     # non-linearity
            nn.Linear(512, 10),                            # 512 -> 10 (digit classes 0..9)
        )                                                  # end MLP

    if name == "lenet":                                   # option 2: classic LeNet-style CNN
        return nn.Sequential(                              # stack CNN layers
            nn.Conv2d(1, 6, kernel_size=5),                # 1x28x28 -> 6x24x24 (5x5 conv, no padding)
            nn.ReLU(),                                     # non-linearity
            nn.AvgPool2d(2),                               # 6x24x24 -> 6x12x12 (downsample by 2)
            nn.Conv2d(6, 16, kernel_size=5),               # 6x12x12 -> 16x8x8
            nn.ReLU(),                                     # non-linearity
            nn.AvgPool2d(2),                               # 16x8x8 -> 16x4x4
            nn.Flatten(),                                  # 16x4x4 -> 256
            nn.Linear(16 * 4 * 4, 120),                    # 256 -> 120
            nn.ReLU(),                                     # non-linearity
            nn.Linear(120, 84),                            # 120 -> 84
            nn.ReLU(),                                     # non-linearity
            nn.Linear(84, 10),                             # 84 -> 10 (classes)
        )                                                  # end LeNet

    if name == "resnet18":                                 # option 3: famous ResNet-18 (adapted to MNIST)
        m = models.resnet18(weights=None)                  # create ResNet-18 without pretrained weights
        m.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)  # change input channels 3->1
        m.fc = nn.Linear(m.fc.in_features, 10)             # change classifier head to 10 classes
        return m                                           # return adapted ResNet-18

    if name == "mobilenet_v2":                             # option 4: famous MobileNetV2 (adapted to MNIST)
        m = models.mobilenet_v2(weights=None)              # create MobileNetV2 without pretrained weights
        first = m.features[0][0]                           # get the very first conv layer in MobileNetV2
        m.features[0][0] = nn.Conv2d(                      # replace it to accept 1-channel input instead of 3
            1,                                             # in_channels = 1 (MNIST)
            first.out_channels,                             # keep original out_channels
            kernel_size=first.kernel_size,                  # keep original kernel size
            stride=first.stride,                            # keep original stride
            padding=first.padding,                          # keep original padding
            bias=False                                      # match typical conv config
        )                                                  # end replacement conv
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 10)  # change classifier head to 10 classes
        return m                                           # return adapted MobileNetV2

    raise ValueError("Unknown model. Choose: mlp, lenet, resnet18, mobilenet_v2")  # error if name not recognised


def train_one_epoch(model, loader, optimiser, loss_fn):    # one pass over the training data
    model.train()                                          # set model to training mode (affects dropout/batchnorm)
    for x, y in loader:                                    # iterate over mini-batches
        x, y = x.to(device), y.to(device)                  # move data to GPU/CPU device
        optimiser.zero_grad()                               # clear old gradients
        loss = loss_fn(model(x), y)                        # forward pass + compute classification loss
        loss.backward()                                    # backpropagate gradients
        optimiser.step()                                   # update model parameters


@torch.no_grad()                                           # disable gradient tracking for evaluation (faster/less memory)
def evaluate(model, loader):                               # compute accuracy on a dataset
    model.eval()                                           # set model to eval mode (affects dropout/batchnorm)
    correct, total = 0, 0                                  # counters for accuracy
    for x, y in loader:                                    # iterate over evaluation batches
        x, y = x.to(device), y.to(device)                  # move batch to device
        pred = model(x).argmax(dim=1)                      # predicted class = index of max logit per sample
        correct += (pred == y).sum().item()                # count correct predictions in this batch
        total += y.numel()                                 # total number of labels in this batch
    return correct / total                                 # final accuracy as a fraction


MODEL_NAME = "resnet18"                                    # choose: "mlp", "lenet", "resnet18", "mobilenet_v2"

model = get_model(MODEL_NAME).to(device)                   # build the chosen model and move it to GPU/CPU
optimiser = optim.Adam(model.parameters(), lr=1e-3)         # Adam optimiser with learning rate 0.001
loss_fn = nn.CrossEntropyLoss()                            # standard loss for multi-class classification

for epoch in range(3):                                     # train for 3 epochs (small demo)
    train_one_epoch(model, train_loader, optimiser, loss_fn) # train for one epoch
    acc = evaluate(model, test_loader)                     # evaluate accuracy on test set
    print(f"Epoch {epoch+1}: test_acc = {acc:.4f}")         # print accuracy each epoch

ImportError: DLL load failed while importing _imaging: The specified module could not be found.